# The dominant protein form of each gene

One objective: for each gene, find the ORF (protein form) that carries most of its expression, and ask whether that protein form differs between the two groups. The unit is the ORF and its transcripts, not the gene. Many transcript isoforms collapse into one ORF, so the ORF is the protein-level view. Run it on SpA against not SpA, and again on B27-positive against B27-negative SpA by changing `paths.json`.

In [ ]:
source("src/heatmap.R")   # read_groups, expression_heatmap, show_heatmap
source("src/paths.R")     # run_locations, figure_stem
suppressPackageStartupMessages(library(data.table))
options(repr.plot.width = 8, repr.plot.height = 4)
loc <- run_locations()
comparison <- loc$comparison
genes <- c("HLA-B", "HLA-C")      # genes to look at closely
alpha <- 0.05                     # FDR for the heatmap of dominant ORFs
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))
levels_g <- unique(groups$group)
table(groups$group)

**Explanation.** `genes` are the genes to examine; `alpha` is the FDR cut for the heatmap (try 0.05, then 0.1). The table shows how many samples are in each group. Nothing here uses the gene-level test: the gene is only used to put the ORFs of one locus side by side.

In [ ]:
orf <- fread(file.path(loc$de_orf, paste0(comparison, "_DE_ORF_edgeR_results.txt")), data.table = FALSE)
orf$gene <- sub("::.*$", "", orf$orf_base_id)                       # gene name is the start of the ORF name
orf$n_isoforms <- lengths(strsplit(orf$orf_all_isoform_id, ","))     # transcripts that collapse into this ORF
cpm <- as.matrix(orf[, paste0(groups$sample, "_cpm")])               # normalised CPM of every sample
rownames(cpm) <- orf$orf_base_id
colnames(cpm) <- groups$sample                                       # name the columns by sample
c(ORFs = nrow(orf), genes = length(unique(orf$gene)), transcripts = sum(orf$n_isoforms))

**Explanation.** The pipeline's ORF results table has, for every ORF, the change between groups (`logFC`, `FDR`), the list of transcripts behind it (`orf_all_isoform_id`) and the normalised CPM in each sample. `n_isoforms` counts the transcripts behind each ORF, which shows how much the protein level collapses the transcript level. The last line counts ORFs, genes and transcripts.

In [ ]:
cls <- fread(Sys.glob(file.path(loc$sqanti_protein, "*best_ORF_SQANTI_classification.tsv"))[1],
             select = c("isoform_id", "tx_cat", "pr_splice_cat"), data.table = FALSE)
# "novel" = not a full splice match; "no ORF" = the isoform has no predicted protein (so no classification)
novelty_of <- function(keys, category) {
  i <- match(keys, cls$isoform_id)
  ifelse(is.na(i), "no ORF", ifelse(cls[[category]][i] != "full-splice_match", "novel", "known"))
}
orf$novel_protein <- novelty_of(orf$orf_base_id, "pr_splice_cat")   # the dominant ORF is a protein form
table(orf$novel_protein)

**Explanation.** The pipeline's SQANTI classification says, for every isoform with a predicted ORF, whether the isoform (`tx_cat`) and its protein (`pr_splice_cat`) are a `full-splice_match` to something already annotated. `novelty_of()` turns that into a label per row: `known`, `novel` (anything that is not a full splice match) or `no ORF`. For isoforms the label is about the isoform; for ORFs it is about the protein. Here it is added to the table as `novel_protein` (about the protein) and drawn as a bar on the heatmaps. The table counts all ORFs by label.

In [ ]:
orf$mean_all <- rowMeans(cpm)                                        # average CPM over all samples
for (g in levels_g) orf[[paste0("mean_", g)]] <- rowMeans(cpm[, groups$sample[groups$group == g], drop = FALSE])
orf$gene_total <- ave(orf$mean_all, orf$gene, FUN = sum)             # all ORFs of the gene together
orf$share <- ifelse(orf$gene_total > 0, orf$mean_all / orf$gene_total, NA)   # fraction of the gene carried by this ORF
# dominant = the ORF with the highest average CPM in its gene (the first one if tied)
is_top <- function(x) seq_along(x) == which.max(x)
orf$dominant <- ave(orf$mean_all, orf$gene, FUN = is_top) == 1
summary(orf$share[orf$dominant])

**Explanation.** For each ORF, `mean_all` is its average normalised CPM over all samples, and `share` is that average divided by the sum over all ORFs of the same gene. The **dominant** ORF of a gene is the one with the highest `mean_all`. A share near 1 means one protein form carries nearly all of the gene; a low share means several forms compete and "dominant" is weak. No cutoff is applied: read the share beside each result. The summary line gives the spread of shares among dominant ORFs.

In [ ]:
top_in <- function(g) {                                              # id of the highest-CPM ORF of each gene, within one group
  m <- orf[[paste0("mean_", g)]]
  ids <- tapply(seq_along(m), orf$gene, function(i) orf$orf_base_id[i[which.max(m[i])]])
  ids
}
dom_1 <- top_in(levels_g[1]); dom_2 <- top_in(levels_g[2])
orf$switch <- orf$dominant & (dom_1[orf$gene] != dom_2[orf$gene])   # gene whose top ORF differs between the groups
table(switch = orf$switch[orf$dominant])

**Explanation.** The dominant ORF is found again within each group alone. A **switch** is a gene whose top ORF in one group is not the top ORF in the other: a change of protein form that a single logFC would not show. The table counts the genes that switch (TRUE) and those that do not. A gene is flagged on its all-samples dominant ORF, so check the two group columns below before reading anything into one.

In [ ]:
sp_file <- paste0(figure_stem("specific_orfs", "tables"), ".tsv")    # from specific_variants.ipynb, if it was run
if (file.exists(sp_file)) {
  sp <- fread(sp_file, data.table = FALSE)
  orf$specific_to <- as.character(sp$specific_to[match(orf$orf_base_id, sp$id)])
} else {
  orf$specific_to <- NA_character_
  message("no ", sp_file, ": run specific_variants.ipynb first to fill specific_to")
}
keep <- c("gene", "orf_base_id", "novel_protein", "n_isoforms", "share", paste0("mean_", levels_g), "logFC", "FDR", "specific_to", "switch")
dominant <- orf[orf$dominant, keep]
dominant <- dominant[order(dominant$FDR), ]
head(dominant, 15)

**Explanation.** One row per gene: its dominant ORF, how many transcripts collapse into it, its share of the gene, its average CPM in each group, `logFC` and `FDR` for that ORF, whether the splice-variant analysis called it specific to one group (`specific_to`, filled when `specific_variants.ipynb` has been run), and whether the gene switches. Sorted by FDR, so the strongest protein-level differences come first.

In [ ]:
sel <- orf[orf$gene %in% genes, c(keep, "dominant")]
sel <- sel[order(sel$gene, -sel$share), ]
rownames(sel) <- NULL
sel

**Explanation.** Every ORF of the genes in `genes`, not only the dominant one, with the same columns, most abundant first within each gene. `dominant` marks the top ORF. This is the protein-level picture for the genes you chose. HLA genes are very variable between people and the reference may map them imperfectly, so check whether a difference follows the allele (for example B27) and not the splicing.

In [ ]:
dir.create("tables", showWarnings = FALSE)
write.table(dominant, paste0(figure_stem("dominant_orf", "tables"), ".tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
write.table(sel, paste0(figure_stem("dominant_orf_chosen_genes", "tables"), ".tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
list.files("tables", pattern = "dominant")

**Explanation.** Both tables are saved in `tables/` with the comparison name.

In [ ]:
sig <- orf$orf_base_id[orf$dominant & !is.na(orf$FDR) & orf$FDR < alpha]
cat(length(sig), "of", sum(orf$dominant), "dominant ORFs have FDR <", alpha, "\n")
show_heatmap(expression_heatmap(cpm[sig, , drop = FALSE], groups, title = paste0("Dominant ORFs, FDR < ", alpha), mean_bar = TRUE, novelty = orf$novel_protein[match(sig, orf$orf_base_id)]), "dominant_orf_heatmap")

**Explanation.** A heatmap of only the dominant ORFs that pass `alpha`. Rows are ORFs, columns are samples split by group, the colour is the row z-score of log2(CPM+1), and the bar at the left is the ORF's average CPM so abundance is not hidden by the colours. If nothing passes, raise `alpha` to 0.1 and say so when you read it.

In [ ]:
chosen <- orf$orf_base_id[orf$gene %in% genes]
show_heatmap(expression_heatmap(cpm[chosen, , drop = FALSE], groups, title = paste(genes, collapse = ", "), mean_bar = TRUE, novelty = orf$novel_protein[match(chosen, orf$orf_base_id)]), "dominant_orf_chosen_genes_heatmap")

**Explanation.** All ORFs of the chosen genes in every sample, whatever their FDR, so you can see which protein forms are present, in which samples, and how abundant each is.

## For the bioinformatician

- **Source of CPM.** The `_cpm` columns of `*_DE_ORF_edgeR_results.txt`, the pipeline's edgeR-normalised values. `logFC` and `FDR` are edgeR's, with Benjamini-Hochberg adjustment (Benjamini and Hochberg 1995).
- **Dominance.** Highest average CPM in the gene, over all samples, with no minimum share. Group sizes differ, so the all-sample average follows the larger group; the per-group tops are used only for the switch flag.
- **Collapse.** One ORF stands for all transcripts that encode the same protein sequence, so a significant ORF can hide many transcript-level differences, and a transcript-level change may leave the ORF unchanged.
- **Limits.** Dominant by average is not dominant in every sample; look at the heatmap. FDR is loosened only for exploration, and the value is in the title.

In [ ]:
si <- Sys.info()
cat("run on :", si[["nodename"]], "(", si[["sysname"]], si[["release"]], ")\n")
cat("date   :", format(Sys.time(), "%Y-%m-%d %H:%M %Z"), "\n")
cat("R      :", R.version.string, "\n")
for (p in c("data.table", "ComplexHeatmap", "jsonlite")) cat(sprintf("  %-16s %s\n", p, as.character(packageVersion(p))))

**Explanation.** Records where and with what versions this ran. It is the same in every notebook.